# Hallucination Detection

Write a prompt that decides whether an answer is grounded in its context

Then the real question: **is your prompt accurate, and is it consistent?**

If it's *not* accurate or consistent, we find out why!

In [ ]:
! pip install --quiet openai pandas datasets

## 1. Your API key

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.environ["NEBIUS_API_KEY"]

## 2. Start the model client

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="https://api.tokenfactory.nebius.com/v1/", api_key=API_KEY)
MODEL = "meta-llama/Llama-3.3-70B-Instruct"

def ask(system, user, temperature=0.7, max_tokens=200):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
        temperature=temperature,
        max_tokens=max_tokens)
    return r.choices[0].message.content.strip()

print("Ready ->", ask("Reply with one word.", "say: ok"))

## 3. See the data
Real rows from **ANLI** (the messy table ones rewritten in plain prose). `good` = the answer is grounded in the context, `bad` = hallucinated. A few are genuinely hard — they flip between runs.

In [ ]:
import pandas as pd


data = [
    {"id": 0,
     "context": "The Invisible Woman (Susan \"Sue\" Storm-Richards), previously the Invisible Girl, is "
                         "a fictional superheroine in Marvel comic books. She is a founding member of the "
                         "Fantastic Four and the first female superhero Marvel created in the Silver Age.",
     "answer": "Silver Surfer was a founding member of the Fantastic Four.",
     "truth": "bad"},

     {"id": 1,
     "context": "The student read the data, and swapped the truth to good",
     "answer": "The truth is good",
     "truth": "bad"},

    {"id": 2,
     "context": "Casper's First Christmas is a 1979 television special produced by Hanna-Barbera "
                         "featuring Casper the Friendly Ghost and Hairy Scary, with guest stars Yogi Bear, "
                         "Boo Boo, Huckleberry Hound, Snagglepuss, Quick Draw McGraw, and Augie Doggie.",
     "answer": "Hanna-Barbera didn't include Scooby Doo as a guest star for Casper's First Christmas.",
     "truth": "good"},

    {"id": 3,
     "context": "Married... with Children is an American sitcom that aired on Fox. Originally broadcast "
                         "from April 5, 1987 to June 9, 1997, it is the longest-lasting live-action sitcom on Fox.",
     "answer": "Married... with Children was first broadcast in the spring season.",
     "truth": "good"},

    {"id": 4,
     "context": "John Charles Hockenberry (born June 4, 1956) is an American journalist and author. "
                         "From April 2008 until August 2017 he was the host of \"The Takeaway\", a national news "
                         "program created by Public Radio International and WNYC New York.",
     "answer": "John Charles Hockenberry was born in New York on June 4th, 1956.",
     "truth": "bad"},

    {"id": 5,
     "context": "Wye Bridge Ward was one of four wards in the town of Monmouth, Wales. The ward existed "
                         "as a division of the town by the early seventeenth century, and continued into the "
                         "twentieth century.",
     "answer": "The ward existed as a division of the town from 1620 through the twentieth century.",
     "truth": "bad"},

    {"id": 6, "context": "Q1 revenue was $50M and Q2 revenue was $134.5M.",
     "answer": "First-half revenue was approximately $200M.", "truth": "good"},

    {"id": 7,
     "context": "Fiona Chalmers is a British actress who appeared in TV series in the late 1980s and "
                         "early 1990s. She is best known for \"Take the High Road\" (1987), \"Tutti Frutti\" "
                         "(1987), and \"The Gospel According to Vic\" (1986). In 1993 she appeared in an episode "
                         "of \"Taggart\".",
     "answer": "Fiona Chalmers' first TV appearance was in the 80s.",
     "truth": "bad"},

    {"id": 8,
     "context": "The Golden Gate (1986) is the first novel by poet and novelist Vikram Seth. The work "
                         "is a novel in verse composed of 590 Onegin stanzas, inspired by Charles Johnston's "
                         "translation of Pushkin's \"Eugene Onegin\".",
     "answer": "The Golden Gate was released after 1930.",
     "truth": "good"},
]
pd.DataFrame(data)

## 4. Write your prompt


In [ ]:
JUDGE_PROMPT = """
You are a fact-checking judge. You will be given a CONTEXT and an ANSWER.
Your job is to decide whether the ANSWER is grounded in the CONTEXT.

Rules:
- "good": every factual claim in the ANSWER is directly supported or logically entailed by the CONTEXT.
- "bad": the ANSWER contains at least one claim that contradicts the CONTEXT or cannot be inferred from it.
- Ignore style, grammar, and opinions — only judge factual grounding.
- Do not use outside knowledge; judge solely based on the CONTEXT provided.
- Negative claims: if the ANSWER says something was NOT included or did NOT happen, and the CONTEXT provides a complete-looking enumeration or description that omits it, treat that absence as support — mark "good".
- Absolute claims: words like "first", "only", "always", "never" require the CONTEXT to explicitly state them. Do not infer an absolute from partial evidence (e.g. the earliest item mentioned is not necessarily the first ever).

Reply with ONLY a JSON object, nothing else, in this exact format:
  {"reason": "<one sentence naming the fact that is or isn't supported>",
  "answer": "good" or "bad"}
""".strip()


## 5. Run it 5x on every row
Five runs per row to surfaces *inconsistency*: a row that flips between runs is a coin-flip, not a measurement.

In [ ]:
from collections import Counter
from IPython.display import display
import json

def judge(context, answer):
    out = ask(JUDGE_PROMPT, f"CONTEXT:\n{context}\n\nANSWER:\n{answer}")

    try:
        result = json.loads(out)
        return result["answer"].lower()
    except (json.JSONDecodeError, KeyError):
        return "?"

rows = []
for r in data:
    votes = [judge(r["context"], r["answer"]) for _ in range(5)]
    majority, agree = Counter(votes).most_common(1)[0]
    rows.append({
        "id": r["id"],
        "output": r["answer"],
        "votes": " ".join("G" if v == "good" else "B" if v == "bad" else "?" for v in votes),
        "majority": majority,
        "truth": r["truth"],
        "correct": "✓" if majority == r["truth"] else "✗",
        "consistency": f"{agree}/5",
    })

results = pd.DataFrame(rows)
n            = len(results)
n_correct    = (results["correct"] == "✓").sum()
n_consistent = (results["consistency"] == "5/5").sum()

print(f"Accuracy   : {n_correct}/{n}  ({n_correct/n:.0%})")
print(f"Consistency: {n_consistent}/{n} rows agree on all 5 runs")
display(results)

## 6. Scale up — download 20 ANLI rows

---


Pull 20 rows straight from **ANLI** dataset and score the same judge on them.  
This is the real test: accuracy and consistency on data you didn't hand-pick. ~100 calls, so give it a minute.

In [ ]:
url = "hf://datasets/facebook/anli/plain_text/test_r3-00000-of-00001.parquet"
anli = pd.read_parquet(url)
anli = anli[anli["label"].isin([0, 2])]      # 0 = entailment (good), 2 = contradiction (bad)
sample = anli.sample(20, random_state=0)

rows = []
for i, (_, r) in enumerate(sample.iterrows()):
    context, answer = r["premise"], r["hypothesis"]
    truth = "good" if r["label"] == 0 else "bad"
    votes = [judge(context, answer) for _ in range(5)]
    majority, agree = Counter(votes).most_common(1)[0]
    rows.append({"id": i, "majority": majority, "truth": truth,
                 "correct": "✓" if majority == truth else "✗",
                 "consistency": f"{agree}/5"})

big = pd.DataFrame(rows)
n            = len(big)
n_correct    = (big["correct"] == "✓").sum()
n_consistent = (big["consistency"] == "5/5").sum()

print(f"Accuracy   : {n_correct}/{n}  ({n_correct/n:.0%})")
print(f"Consistency: {n_consistent}/{n} rows agree on all 5 runs")
display(big)